# Set Up

In [1]:
import pandas as pd
from pcj.utils import dados, BASE_DIR
from pcj.modelos_ML import random_forest, knn_regressor

In [2]:
df, metadados, variaveis = dados(r"data\processed\dados_med_tratados.xlsx")

# Blocos seguidos × divisões aleatórias
Nos notebooks `ML_outliers_*`, a validação cruzada usa 5 blocos de meses seguidos: cada bloco de teste é um período que o modelo não viu no treino. Numa divisão aleatória (`embaralhar=True`), os meses são sorteados para os blocos, e um mês de teste pode ter os meses vizinhos no treino. Este notebook mede a diferença entre as duas divisões na base tratada, com os mesmos modelos: RF (pH) e KNN (as 16 variáveis).

## Blocos seguidos
Os resultados com blocos seguidos vêm do Excel gravado pelo notebook `ML_outliers_tratados` (as mesmas funções, com o padrão `embaralhar=False`; se algo mudar lá, rode aquele antes deste).

In [3]:
caminho_blocos = BASE_DIR / "data" / "output" / "ML_outliers_tratados.xlsx"
blocos_rf = pd.read_excel(caminho_blocos, sheet_name="rf")
blocos_knn = pd.read_excel(caminho_blocos, sheet_name="knn")
blocos_knn[["variavel", "rmse", "mae", "r2"]]

,variavel,rmse,mae,r2
0,COR,78.918852,37.885344,0.496810
1,TURB.,50.239894,33.599702,0.468770
2,pH,0.141134,0.111863,0.130892
3,ALC.,13.295878,9.776063,0.553414
4,AC.,5.313244,3.623009,-0.135784
5,O.C.,2.496647,1.822802,-0.113796
6,O.D.,0.753419,0.598298,0.196735
7,Cl,9.698348,7.076877,0.474154
8,DUR.,13.044501,9.727684,0.201696
9,Fe,1.431252,0.865556,0.421603


## Divisões aleatórias
30 rodadas com `embaralhar=True`, uma para cada seed do sorteio (`seed_embaralhar` de 0 a 29), porque um sorteio só pode ser sorte ou azar. Em cada rodada, os 118 meses completos são sorteados para os 5 blocos, e as métricas juntam os 5 blocos, como nos blocos seguidos. O KNN leva cerca de 40 s por rodada (uns 20 minutos no total).

In [4]:
# Uma linha por rodada (seed), modelo e variável prevista
linhas = []

for seed in range(30):
    resultado_rf = random_forest(df, variaveis, embaralhar=True, seed_embaralhar=seed)
    resultado_knn = knn_regressor(df, variaveis, embaralhar=True, seed_embaralhar=seed)

    # Cada modelo: nome e tabela de resultados desta rodada
    for modelo, resultado in [("rf", resultado_rf), ("knn", resultado_knn)]:
        # iterrows percorre a tabela linha por linha (no RF, uma linha; no KNN, uma por variável)
        for _, linha in resultado.iterrows():
            linhas.append({
                "modelo": modelo,
                "seed": seed,
                "variavel": linha["variavel"],
                "rmse": linha["rmse"],
                "mae": linha["mae"],
                "r2": linha["r2"]
            })

rodadas = pd.DataFrame(linhas)
print("Rodadas:", rodadas["seed"].nunique(), "| linhas:", len(rodadas))

Rodadas: 30 | linhas: 510


## Comparação
Para cada modelo, variável prevista e métrica: o valor com blocos seguidos, a média e o desvio padrão das 30 divisões aleatórias, a diferença (aleatório − blocos) e em quantas das 30 rodadas o aleatório saiu melhor.
- RMSE e MAE: diferença negativa = o aleatório errou menos; "melhor" = erro menor.
- R²: o contrário; diferença positiva = o aleatório foi melhor; "melhor" = R² maior.

In [5]:
# Cada modelo: nome e resultado com blocos seguidos
modelos = [("rf", blocos_rf), ("knn", blocos_knn)]
linhas = []

for modelo, res_blocos in modelos:
    # set_index deixa a variável prevista como índice, para buscar o valor de cada uma pelo nome
    res_blocos = res_blocos.set_index("variavel")
    rodadas_modelo = rodadas[rodadas["modelo"] == modelo]

    for variavel in res_blocos.index:
        # As 30 rodadas desta variável
        rodadas_var = rodadas_modelo[rodadas_modelo["variavel"] == variavel]

        for metrica in ["rmse", "mae", "r2"]:
            blocos = res_blocos.at[variavel, metrica]
            valores = rodadas_var[metrica]

            # Rodadas em que o aleatório saiu melhor: R² maior; nas outras métricas, erro menor
            if metrica == "r2":
                melhor = (valores > blocos).sum()
            else:
                melhor = (valores < blocos).sum()

            linhas.append({
                "modelo": modelo,
                "metrica": metrica,
                "variavel": variavel,
                "blocos": blocos,
                "aleatorio_media": valores.mean(),
                "aleatorio_dp": valores.std(),     # DP entre as 30 rodadas (ddof=1, o padrão do pandas)
                "diferenca": valores.mean() - blocos,
                "rodadas_aleatorio_melhor": melhor
            })

comparacao = pd.DataFrame(linhas)

Uma tabela por modelo e métrica, com as variáveis previstas nas linhas (valores arredondados para 3 casas só na exibição; o Excel guarda os números inteiros).

In [6]:
for modelo in ["rf", "knn"]:
    for metrica in ["rmse", "mae", "r2"]:
        # Linhas deste modelo e desta métrica, com a variável como índice
        tabela = comparacao[(comparacao["modelo"] == modelo) & (comparacao["metrica"] == metrica)]
        tabela = tabela.set_index("variavel").drop(columns=["modelo", "metrica"])

        print(modelo.upper(), "-", metrica.upper())
        display(tabela.round(3))

RF - RMSE


,blocos,aleatorio_media,aleatorio_dp,diferenca,rodadas_aleatorio_melhor
variavel,,,,,
pH,0.13,0.128,0.004,-0.002,21


RF - MAE


,blocos,aleatorio_media,aleatorio_dp,diferenca,rodadas_aleatorio_melhor
variavel,,,,,
pH,0.104,0.101,0.003,-0.003,23


RF - R2


,blocos,aleatorio_media,aleatorio_dp,diferenca,rodadas_aleatorio_melhor
variavel,,,,,
pH,0.26,0.283,0.05,0.023,21


... (2 outputs left out)

KNN - MAE


,blocos,aleatorio_media,aleatorio_dp,diferenca,rodadas_aleatorio_melhor
variavel,,,,,
COR,37.885,33.543,2.273,-4.342,27
TURB.,33.600,30.308,0.879,-3.292,30
pH,0.112,0.102,0.003,-0.010,30
ALC.,9.776,8.044,0.272,-1.733,30
AC.,3.623,3.304,0.106,-0.319,30
O.C.,1.823,1.562,0.041,-0.261,30
O.D.,0.598,0.489,0.011,-0.109,30
Cl,7.077,6.257,0.189,-0.820,30
DUR.,9.728,8.236,0.339,-1.492,30


KNN - R2


,blocos,aleatorio_media,aleatorio_dp,diferenca,rodadas_aleatorio_melhor
variavel,,,,,
COR,0.497,0.567,0.044,0.070,28
TURB.,0.469,0.565,0.027,0.096,30
pH,0.131,0.251,0.037,0.120,30
ALC.,0.553,0.682,0.024,0.129,30
AC.,-0.136,0.016,0.052,0.151,30
O.C.,-0.114,0.011,0.039,0.125,30
O.D.,0.197,0.415,0.020,0.218,30
Cl,0.474,0.550,0.021,0.075,30
DUR.,0.202,0.393,0.038,0.191,30


# Gravação
Abas `rodadas` (uma linha por modelo × seed × variável) e `comparacao` (uma linha por modelo × métrica × variável). Rodar de novo dá os mesmos números: as seeds do sorteio são fixas (0 a 29), o RF usa random_state=42 e o KNN não tem sorteio.

In [7]:
caminho = BASE_DIR / "data" / "output" / "ML_blocos_aleatorio.xlsx"

# ExcelWriter permite gravar várias abas no mesmo arquivo
with pd.ExcelWriter(caminho) as arquivo:
    rodadas.to_excel(arquivo, sheet_name="rodadas", index=False)
    comparacao.to_excel(arquivo, sheet_name="comparacao", index=False)

print("Gravado em:", caminho)

Gravado em: C:\_Master\Programacao\Guilherme-Betta\1_Projects\IC2\IC2-item-3\data\output\ML_blocos_aleatorio.xlsx
